# 04 — Multi-Document RAG: Study Assistant

A knowledge base built from **4 different topic files**, with **metadata-filtered** retrieval.

### New concepts
| Concept | Description |
|---------|-------------|
| Multi-document ingestion | Load all `.txt` files with topic metadata tags |
| ChromaDB | Persistent vector store with rich metadata support |
| Metadata filtering | Retrieve only from a specific topic (`where={"topic": "space"}`) |
| Multi-topic Q&A | Ask questions across all topics; the system routes to the right context |

> **Requires:** `GROQ_API_KEY` in a `.env` file

## 1. Install Dependencies

In [ ]:
# !pip install langchain langchain-community langchain-groq langchain-huggingface chromadb python-dotenv

## 2. Imports & Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 3. Load All Four Topic Files

We tag each document with a `topic` and `category` in the metadata.
This lets us filter retrieval to a specific subject later.

In [ ]:
DATA_DIR = Path("../data")

# Map each file to a topic tag and a short category
TOPIC_MAP = {
    "space_facts.txt":    {"topic": "space",     "category": "science"},
    "nutrition_guide.txt":{"topic": "nutrition", "category": "health"},
    "python_tips.txt":    {"topic": "python",    "category": "programming"},
    "world_history.txt":  {"topic": "history",   "category": "humanities"},
}

all_docs = []
for filename, meta in TOPIC_MAP.items():
    filepath = DATA_DIR / filename
    text = filepath.read_text(encoding="utf-8")
    doc  = Document(page_content=text, metadata={"source": filename, **meta})
    all_docs.append(doc)
    print(f"  Loaded {filename:25s} ({len(text):,} chars) → topic: {meta['topic']}")

print(f"\nTotal documents loaded: {len(all_docs)}")

## 4. Split All Documents into Chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_documents(all_docs)
print(f"Total chunks created: {len(chunks)}")

# Show topic distribution
from collections import Counter
topic_counts = Counter(c.metadata["topic"] for c in chunks)
for topic, count in sorted(topic_counts.items()):
    print(f"  {topic:15s}: {count} chunks")

## 5. Build ChromaDB Vector Store

ChromaDB stores vectors and their metadata in a persistent local database.
This makes it ideal for filtering — you can query with `where` conditions.

In [ ]:
print("Loading embedding model...")
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Building ChromaDB vector store...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedder,
    collection_name="study_assistant",
    persist_directory="./chroma_study"
)
print(f"ChromaDB ready — {vectorstore._collection.count()} vectors stored")

## 6. Standard Retrieval (all topics)

Without a filter, the retriever searches across all topics.

In [ ]:
retriever_all = vectorstore.as_retriever(search_kwargs={"k": 4})

query = "How do I write efficient code?"
results = retriever_all.invoke(query)
print(f"Query: '{query}'")
print(f"Returned {len(results)} chunks from topics: {[r.metadata['topic'] for r in results]}")
for r in results:
    print(f"  [{r.metadata['topic']:10s}] {r.page_content[:90]}...")

## 7. Metadata-Filtered Retrieval

Use `filter=` to restrict retrieval to a specific topic.
This is essential when you know which knowledge domain to search.

In [ ]:
def topic_retriever(topic: str, k: int = 3):
    """Returns a retriever that only searches within one topic."""    return vectorstore.as_retriever(
        search_kwargs={"k": k, "filter": {"topic": topic}}
    )

# Same query, filtered to Python only
query = "How do I write efficient code?"
py_results = topic_retriever("python").invoke(query)
print(f"Filtered to 'python' — topics returned: {[r.metadata['topic'] for r in py_results]}")
for r in py_results:
    print(f"  {r.page_content[:120]}...")

## 8. Multi-Topic Q&A with Source Citations

In [ ]:
STUDY_PROMPT = PromptTemplate.from_template(
    """You are a knowledgeable study assistant. Answer the question based strictly on the context.
Cite which topic/source the answer comes from at the end.

Context:
{context}

Question: {question}

Answer (include a 'Source:' line at the end):"""
)

llm = ChatGroq(model="llama3-8b-8192", temperature=0.1)

def format_docs_with_meta(docs):
    return "\n\n".join(
        f"[Topic: {d.metadata['topic']} | Source: {d.metadata['source']}]\n{d.page_content}"
        for d in docs
    )

def study_qa(question: str, topic_filter: str = None) -> dict:
    """Ask a question, optionally restricting to one topic."""    retriever = topic_retriever(topic_filter) if topic_filter else retriever_all
    docs = retriever.invoke(question)
    context = format_docs_with_meta(docs)

    rag_chain = (
        {"context": lambda _: context, "question": RunnablePassthrough()}
        | STUDY_PROMPT
        | llm
        | StrOutputParser()
    )
    answer  = rag_chain.invoke(question)
    sources = list({d.metadata["source"] for d in docs})
    topics  = list({d.metadata["topic"]  for d in docs})
    return {"answer": answer, "sources": sources, "topics": topics}

## 9. Ask Questions Across Topics

In [ ]:
# Cross-topic questions — searches all 4 knowledge bases
cross_topic_questions = [
    ("What were the main effects of the Industrial Revolution?", None),
    ("What vitamins support immune function?",                   None),
    ("How does Python handle memory-efficient iteration?",       None),
    ("What was the Big Bang?",                                   None),
]

for question, topic_filter in cross_topic_questions:
    result = study_qa(question, topic_filter)
    print(f"\n{'═'*58}")
    print(f"  Q: {question}")
    print(f"  Topics searched: {result['topics']}")
    print(f"  Sources: {result['sources']}")
    print(f"{'─'*58}")
    print(f"  {result['answer'][:300]}...")

In [ ]:
# Topic-specific questions — filtered retrieval
print("\n" + "="*58)
print("  TOPIC-FILTERED QUERIES")
print("="*58)

filtered_questions = [
    ("What is the difference between supervised and unsupervised learning?", "history"),  # wrong topic
    ("What is dietary fibre?",                                               "nutrition"),
    ("How do decorators work?",                                              "python"),
]

for question, topic in filtered_questions:
    result = study_qa(question, topic)
    print(f"\n  Q (topic={topic}): {question}")
    print(f"  A: {result['answer'][:200]}...")

## 10. Key Takeaways

| Concept | What you learned |
|---------|-----------------|
| Metadata tagging | Add `topic`, `category`, `source` to every chunk |
| ChromaDB | Persistent vector store with native metadata filtering |
| `filter=` param | Restrict retrieval to a specific topic at query time |
| Multi-doc ingestion | Load many files in a loop; chunk + tag each one |
| Source citations | Track which document each answer came from |

**Next notebook →** add agentic intelligence — let the graph decide whether to retrieve and retry on poor results.